# Lab 1 — Count before you send (EP07)

From the theory video: the context window has a fixed size (C) and five parts compete for it — **C ≥ S + M + T + H + R** (Equation 18.1).
Send more than that and an API may drop part of it **without any error** (silent truncation).

**What you'll learn**
1. Count tokens with Gemma's own tokenizer, and compare with the "4 characters ≈ 1 token" rule
2. Assemble a prompt from blocks (Equation 18.8) and count each block — does our count match the API's?
3. A pre-flight check: compare with C − R before every call
4. Actually send a prompt longer than the context — the first time we get an error, the second time no error at all

The papers in the long conversation are **simulated data generated for teaching**. Every token count, error and model answer comes from a real run.

> Source: Haggai Roitman, *The Hitchhiker's Guide to Agentic AI*, §18.2–18.3 — CC BY-SA 4.0

> ✏️ **Starter notebook — you write the key parts**
>
> Most of the code is ready. Write only the **4 places marked `✏️ TODO`** (replace `___` or `...` with your code) — each one gives the time in the video where that code is explained (▶ m:ss). Watch the video first, then try it yourself without opening the solution.
>
> Run every cell from top to bottom. A `NameError` or `TypeError` in a cell you have not finished yet is normal.
>
> Stuck? The solution is **`ep07-lab1-count-before-send.en.ipynb`** (the same code as in the video).

## 0. Setup

| You need | Details |
|---|---|
| Python 3.10 or later | or Google Colab, which already has it |
| An OpenRouter API key | sign up at openrouter.ai → Keys → Create Key, then add credit (this lab costs about 1–2 US cents) |
| Packages | `openai`, `python-dotenv`, `tokenizers` (installed in the next cell) |

**Adding the API key**
- **Colab:** the 🔑 Secrets icon → name `OPENROUTER_API_KEY` → turn on *Notebook access*
- **Your own machine:** create a `.env` file next to the notebook with the line `OPENROUTER_API_KEY=sk-or-...`

⚠️ Never put the key in the code, and never share your `.env` file

In [ ]:
%pip install -q "openai>=1.50" python-dotenv "tokenizers==0.23.3"

## 1. Calling Gemma through OpenRouter

**Why `google/gemma-4-26b-a4b-it`** — it follows instructions in Thai well, **its tokenizer is open to download** (we can count tokens on our machine exactly as the model does),
and it can call tools, which Lab 4 needs

**Why the provider is pinned to DeepInfra** — the same one as EP03 and EP06 · it accepts a context of **262,144 tokens** (this lab's C)
Providers accept different context sizes (some 131,072); without the pin, C would change whenever OpenRouter switches provider

**Not free**, but very cheap: DeepInfra charges USD 0.07 per million input tokens and USD 0.34 per million output tokens
— at the end of the lab we add up the tokens actually used
· The `:free` version doesn't work (tested 2026-11-16: 3 out of 3 calls got 429 "rate-limited upstream")

In [ ]:
import json, os, random
from openai import OpenAI, BadRequestError

def load_key():
    try:  # Google Colab Secrets
        from google.colab import userdata
        return userdata.get("OPENROUTER_API_KEY")
    except ImportError:
        pass
    from dotenv import load_dotenv
    load_dotenv()
    return os.environ["OPENROUTER_API_KEY"]

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=load_key())

In [ ]:
MODEL = "google/gemma-4-26b-a4b-it"
PROVIDER = {"order": ["deepinfra/fp8"], "allow_fallbacks": False}
EXTRA = {"reasoning": {"enabled": False}, "provider": PROVIDER}

USAGE = {"in": 0, "out": 0}

def chat(messages, **kw):
    reply = client.chat.completions.create(model=MODEL, messages=messages, max_tokens=300, extra_body=EXTRA | kw)
    USAGE["in"] += reply.usage.prompt_tokens
    USAGE["out"] += reply.usage.completion_tokens
    return reply

## 2. Counting tokens with Gemma's own tokenizer

The textbook (§18.2.6): count with **that model's own** tokenizer, not a word count or the length divided by 4
— the "4 characters per token" rule can be off by 20–40% for code, JSON and non-English text

In EP01 we used OpenAI's `tiktoken` for the big picture; this time we use Gemma's real tokenizer (about 30 MB the first time)

In [ ]:
os.environ["HF_HUB_VERBOSITY"] = "error"  # hide the "no HF_TOKEN" notice: the tokenizer is public
from tokenizers import Tokenizer

tok = Tokenizer.from_pretrained(MODEL)
print("vocabulary:", f"{tok.get_vocab_size():,}", "pieces")

# ✏️ TODO 1 (▶ video 3:10): count the tokens of text with Gemma's tokenizer (no special tokens)

def count_tokens(text):
    return len(___)

In [ ]:
SAMPLES = {
    "English": "The context window is the agent's working memory. Every token in the window costs money and latency; every token not in the window is invisible to the model.",
    "Thai": "ช่วยสรุปงานวิจัยเรื่องการพยากรณ์ข้อมูลอนุกรมเวลาด้วยทรานส์ฟอร์มเมอร์ให้หน่อยครับ อยากได้สามย่อหน้า",
    "Python": 'for i, msg in enumerate(history):\n    if msg["role"] == "tool":\n        weights[i] = 4\n',
    "JSON": json.dumps({"id": "P07", "title": "PatchTST", "year": 2023, "scores": [0.412, 0.387, 0.355]}),
}

for kind, text in SAMPLES.items():
    real, guess = count_tokens(text), len(text) / 4
    print(f"{kind:8} {len(text):4} chars → Gemma {real:3} tokens · 4-chars rule {guess:5.0f} ({(guess - real) / real:+.0%})")

## 3. Assembling a prompt from blocks (Equation 18.8)

**Prompt = Concat(SystemBlock, MemoryBlock, ToolBlock, HistoryBlock, QueryBlock)**

- **System** — the four-section template from §18.3.1: Identity · Capabilities · Constraints · Output Format
- **Memory** — what the harness remembers for the agent (fixed in this lab — memory has its own episode, EP12)
- **Tools** — tool descriptions (written the EP06 Lab 1 way)
- **History** and **Query** — the conversation so far and the latest question

Each block is its own variable, so one can change without touching the others

In [ ]:
SYSTEM_TEMPLATE = """# Identity
You are {agent_name}, a research assistant built by Cogito Learn. Today's date is {date}.
# Capabilities
You can use these tools: {tool_list}.
# Constraints
- Never invent a paper. Cite the paper id for every claim.
# Output Format
Answer in Thai, in at most three short paragraphs."""

MEMORY = "Known about the user: master's student; writing a literature review on transformers for time series."
TOOLS = ("search_papers(query, year_from): search the paper library by topic; returns [{id, title, year}]. "
         "Use for any question about papers. Do NOT use for questions about the user.")
HISTORY = [
    {"role": "user", "content": "Which papers have I already read?"},
    {"role": "assistant", "content": "So far: Informer (P01) and PatchTST (P05)."},
]
QUERY = "What should I read next, and why?"

# ✏️ TODO 2 (▶ video 5:02): join the blocks in Equation 18.8 order: system message → history → query

def assemble(system, memory, tools, history, query):
    system_text = f"{system}\n\n# Memory\n{memory}\n\n# Tools\n{tools}"
    return ___

SYSTEM = SYSTEM_TEMPLATE.format(agent_name="Cogito", date="2026-11-16", tool_list="search_papers")
messages = assemble(SYSTEM, MEMORY, TOOLS, HISTORY, QUERY)
print(messages[0]["content"])

### Counting the way the API counts

The API doesn't count just the text — every message is wrapped in Gemma's chat format
(`<|turn>user` … `<turn|>`), and at the very end comes the header of the reply, `<|turn>model`, which the model continues
So we count each message in that same format

In [ ]:
def message_tokens(m):
    role = "model" if m["role"] == "assistant" else m["role"]
    return count_tokens(f"<|turn>{role}\n{m['content'].strip()}<turn|>\n")

REPLY_HEADER = count_tokens("<bos><|turn>model\n<|channel>thought\n<channel|>")

def count_messages(messages):
    return sum(message_tokens(m) for m in messages) + REPLY_HEADER

sent = count_messages(messages)
reply = chat(messages)
print("we counted:", sent, "· the API counted:", reply.usage.prompt_tokens)
print(reply.choices[0].message.content)

## 4. The pre-flight check (§18.2.6)

Before every LLM call:
1. Count the tokens of the assembled prompt with the model's tokenizer
2. Compare with **C − R** (context size − space reserved for the answer)
3. If it's over: compress, truncate, or **stop with an explicit error**
4. Log the token count of each part

The textbook's fixed budget (Equation 18.2): S 10% · M 20% · T 10% · H 50% · R 10% of C

In [ ]:
C = 262_144          # DeepInfra's context for this model
SHARE = {"S system": 0.10, "M memory": 0.20, "T tools": 0.10, "H history+query": 0.50, "R reserved": 0.10}
R = int(SHARE["R reserved"] * C)

class ContextOverflow(Exception):
    pass

def preflight(system, memory, tools, history, query):
    parts = {"S system": count_tokens(system), "M memory": count_tokens(memory), "T tools": count_tokens(tools),
             "H history+query": sum(message_tokens(m) for m in history) + count_tokens(query)}
    messages = assemble(system, memory, tools, history, query)
    total = count_messages(messages)
    for name, n in parts.items():
        print(f"  {name:16} {n:>8,} tokens ({n / C:6.2%} of C, budget {SHARE[name]:.0%})")
    print(f"  {'total':16} {total:>8,} / limit C - R = {C - R:,}")
    # ✏️ TODO 3 (▶ video 7:05): if total is over C − R, raise ContextOverflow stating the token count and the limit
    if ___:
        raise ___
    return messages

preflight(SYSTEM, MEMORY, TOOLS, HISTORY, QUERY);

## 5. A conversation longer than the context

The user pastes papers one by one into the research assistant (**simulated papers**, randomly generated) and, **in the middle of the conversation**, mentions one thing:
the review's due date and the advisor's name — then the last message asks about it

We try three ways: ① through the pre-flight ② sent as is ③ sent with OpenRouter cutting out the middle for us (`middle-out`)

In [ ]:
random.seed(7)
TOPICS = ["forecasting", "anomaly detection", "imputation", "classification", "traffic", "energy load", "weather", "finance"]
METHODS = ["patching", "sparse attention", "frequency decomposition", "channel independence", "state space layers", "diffusion"]

def fake_paper(k):  # simulated data for teaching: ~590 tokens per paper
    t, m = random.choice(TOPICS), random.choice(METHODS)
    lines = [f"Paper {k}: A study of {m} for time-series {t}."]
    for j in range(14):
        lines.append(f"In experiment {j + 1}, the {m} variant reached an error of {random.uniform(0.1, 0.9):.3f} "
                     f"against a baseline of {random.uniform(0.2, 1.0):.3f}, using a window of {random.choice([96, 192, 336, 512])} points.")
    return " ".join(lines)

In [ ]:
LONG = [{"role": "user", "content": "I'll paste papers one by one; just reply 'noted'."},
        {"role": "assistant", "content": "noted"}]
for k in range(1, 461):
    LONG += [{"role": "user", "content": fake_paper(k)}, {"role": "assistant", "content": "noted"}]
    if k == 230:  # the middle of the conversation
        LONG += [{"role": "user", "content": "Side note: my review is due on 12 December and my advisor is Dr. Kanya."},
                 {"role": "assistant", "content": "noted"}]
ASK = "When is my review due, and who is my advisor?"
print(len(LONG), "messages")

In [ ]:
try:
    preflight(SYSTEM, MEMORY, TOOLS, LONG, ASK)
except ContextOverflow as e:
    print("🛑 pre-flight:", e)

In [ ]:
big = assemble(SYSTEM, MEMORY, TOOLS, LONG, ASK)
print("we would send:", f"{count_messages(big):,}", "tokens")
try:
    chat(big)
except BadRequestError as e:
    print("❌", json.loads(e.body["metadata"]["raw"])["error"]["message"])

In [ ]:
reply = chat(big, transforms=["middle-out"])
print("we sent:", f"{count_messages(big):,}", "· the model saw:", f"{reply.usage.prompt_tokens:,}")
print(reply.choices[0].message.content)

In [ ]:
# ✏️ TODO 4 (▶ video 10:12): warn when the model saw fewer tokens than we sent (allow 5 tokens)
def check_seen(sent, reply):
    seen = ___
    if ___:
        print(f"⚠️ silent truncation: sent {sent:,} tokens, the model saw {seen:,} ({sent - seen:,} dropped)")

check_seen(count_messages(big), reply)

In [ ]:
cost = USAGE["in"] / 1e6 * 0.07 + USAGE["out"] / 1e6 * 0.34
print(f"the whole lab used {USAGE['in']:,} input tokens + {USAGE['out']:,} output tokens ≈ ${cost:.4f}")

## Summary

| What we did | What we saw in the lab |
|---|---|
| Count with the model's tokenizer | The 4-character rule guesses too many for English but too few for Thai, code and JSON (JSON by more than half) |
| Assemble the prompt from blocks | Counting each message in Gemma's chat format → matches `usage.prompt_tokens` within 1 token |
| Pre-flight check | The long conversation is stopped before sending, with a log showing the history is over its budget |
| Sent as is | DeepInfra refuses: 400 "Input too long" |
| `middle-out` | No error · the model saw only ~175K of ~271K tokens · the message in the middle is gone → it says the information isn't there |

**Exercises**
1. Move the "Side note" message to right after paper 3 (early in the conversation) and run way three again — can the model answer now, and why?
2. Extend `preflight` to say at least how many history tokens must go for the prompt to fit

**Next — Lab 2:** when the history outgrows the window — comparing FIFO, pinning, summarizing and importance weights